In [1]:
from google.colab import files
import urllib.request
import subprocess
import sys

code_url = "https://raw.githubusercontent.com/msim0039-bit/mma3001-sensor-project/main/src/audit.py"
urllib.request.urlretrieve(code_url, "audit.py")

uploaded = files.upload()
zip_name = next(iter(uploaded))

subprocess.run([sys.executable, "audit.py", zip_name], check=True)

Saving 5EnvSensor_MayToDec2024_180kRows.zip to 5EnvSensor_MayToDec2024_180kRows.zip


CompletedProcess(args=['/usr/bin/python3', 'audit.py', '5EnvSensor_MayToDec2024_180kRows.zip'], returncode=0)

In [2]:
result = subprocess.run(
    [sys.executable, "audit.py", zip_name],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

Sensor ID: 6012002000869
CSV rows for sensor: 29341
Distinct timestamp/value pairs: 13800
Distinct measurement timestamps: 13800
Timestamps with conflicting CO2 values: 0
Unit: ppm



In [3]:
import csv
import io
import json
import zipfile
from datetime import datetime, timezone
import numpy as np

readings = {}

with zipfile.ZipFile(zip_name) as archive:
    csv_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    )

    with archive.open(csv_name) as raw_file:
        rows = csv.DictReader(io.TextIOWrapper(raw_file, encoding="utf-8-sig"))

        for row in rows:
            if row["sensorid"] != "6012002000869":
                continue

            for measurement in json.loads(row["jsondata"]):
                if measurement.get("variable", {}).get("name") == "Carbon dioxide":
                    for item in measurement.get("values", []):
                        readings[int(item["time"])] = float(item["value"])

times = np.array(sorted(readings))
co2 = np.array([readings[time] for time in times])
intervals_minutes = np.diff(times) / 60

print("First measurement (UTC):", datetime.fromtimestamp(int(times[0]), timezone.utc))
print("Last measurement (UTC): ", datetime.fromtimestamp(int(times[-1]), timezone.utc))
print("Number of readings:", len(co2))
print("CO2 range (ppm):", co2.min(), "to", co2.max())
print("Median interval (minutes):", np.median(intervals_minutes))
print("90th percentile interval (minutes):", np.percentile(intervals_minutes, 90))
print("Largest interval (minutes):", intervals_minutes.max())

First measurement (UTC): 2024-02-29 06:04:56+00:00
Last measurement (UTC):  2024-06-13 17:45:44+00:00
Number of readings: 13800
CO2 range (ppm): 403.0 to 646.0
Median interval (minutes): 10.0
90th percentile interval (minutes): 19.953333333333344
Largest interval (minutes): 29.983333333333334


In [4]:
# Keep the actual readings as ground truth, but hide six from the methods.
start = 1000
hidden = slice(start, start + 6)

true_values = co2[hidden]
hidden_times = times[hidden]

left_time, right_time = times[start - 1], times[start + 6]
left_value, right_value = co2[start - 1], co2[start + 6]

# Baseline: carry the last available reading forward.
previous_value = np.full(6, left_value)

# Linear interpolation: use both readings around the gap.
linear = left_value + (
    (hidden_times - left_time) / (right_time - left_time)
) * (right_value - left_value)

print("Actual CO2 (ppm):          ", true_values)
print("Previous-value estimate:  ", previous_value)
print("Linear estimate (ppm):    ", np.round(linear, 1))
print("Previous-value MAE (ppm): ", np.mean(np.abs(true_values - previous_value)))
print("Linear MAE (ppm):         ", np.mean(np.abs(true_values - linear)))

Actual CO2 (ppm):           [470. 462. 475. 455. 466. 470.]
Previous-value estimate:   [446. 446. 446. 446. 446. 446.]
Linear estimate (ppm):     [446.2 446.4 446.9 447.1 447.3 447.8]
Previous-value MAE (ppm):  20.333333333333332
Linear MAE (ppm):          19.37037037037037


In [5]:
# Reserve the final 25% of readings for this evaluation.
test_start = int(0.75 * len(co2))

for gap_size in (6, 18, 36):
    previous_errors = []
    linear_errors = []
    gap_count = 0

    # Move past each tested window so hidden readings do not overlap.
    for start in range(test_start + 1, len(co2) - gap_size, gap_size + 8):
        window_times = times[start - 1:start + gap_size + 1]

        # Skip windows containing an existing break longer than 30 minutes.
        if np.any(np.diff(window_times) > 30 * 60):
            continue

        actual = co2[start:start + gap_size]
        target_times = times[start:start + gap_size]

        left_time = times[start - 1]
        right_time = times[start + gap_size]
        left_value = co2[start - 1]
        right_value = co2[start + gap_size]

        previous = np.full(gap_size, left_value)
        interpolated = left_value + (
            (target_times - left_time) / (right_time - left_time)
        ) * (right_value - left_value)

        previous_errors.extend(np.abs(actual - previous))
        linear_errors.extend(np.abs(actual - interpolated))
        gap_count += 1

    print(
        f"{gap_size} hidden readings | {gap_count} gaps | "
        f"previous MAE: {np.mean(previous_errors):.2f} ppm | "
        f"linear MAE: {np.mean(linear_errors):.2f} ppm"
    )

6 hidden readings | 246 gaps | previous MAE: 11.02 ppm | linear MAE: 8.14 ppm
18 hidden readings | 132 gaps | previous MAE: 14.02 ppm | linear MAE: 9.57 ppm
36 hidden readings | 78 gaps | previous MAE: 15.72 ppm | linear MAE: 11.61 ppm


In [6]:
import urllib.request
from datetime import datetime, timedelta

url = "https://raw.githubusercontent.com/msim0039-bit/mma3001-sensor-project/main/src/reconstruct.py"
urllib.request.urlretrieve(url, "reconstruct.py")

from reconstruct import reconstruct_gap

start = datetime(2024, 3, 1, 12, 0)
end = start + timedelta(minutes=30)
missing = [start + timedelta(minutes=10), start + timedelta(minutes=20)]

print("Previous:", reconstruct_gap(start, 400, missing, end, 430, "previous"))
print("Linear:", reconstruct_gap(start, 400, missing, end, 430, "linear"))

Previous: [400.0, 400.0]
Linear: [410.0, 420.0]


In [7]:
from datetime import datetime, timedelta
import numpy as np

# Hide six real readings, starting at position 1000.
start_index = 1000
gap_size = 6

def as_datetime(seconds):
    return datetime(1970, 1, 1) + timedelta(seconds=float(seconds))

left_time = as_datetime(times[start_index - 1])
right_time = as_datetime(times[start_index + gap_size])
hidden_times = [
    as_datetime(t) for t in times[start_index:start_index + gap_size]
]

left_value = co2[start_index - 1]
right_value = co2[start_index + gap_size]
actual = co2[start_index:start_index + gap_size]

for method in ("previous", "linear"):
    predicted = reconstruct_gap(
        left_time, left_value, hidden_times, right_time, right_value, method
    )
    mae = np.mean(np.abs(actual - predicted))
    print(f"{method}: MAE = {mae:.2f} ppm")

previous: MAE = 20.33 ppm
linear: MAE = 19.37 ppm


In [8]:
from time import perf_counter

evaluation_start = int(0.50 * len(co2))
evaluation_end = int(0.75 * len(co2))

for gap_size in (6, 18, 36):
    errors = {"previous": [], "linear": []}
    runtimes = {"previous": 0.0, "linear": 0.0}
    gap_count = 0

    # Leave space between tested gaps so their hidden readings do not overlap.
    for start in range(
        evaluation_start + 1,
        evaluation_end - gap_size,
        gap_size + 8,
    ):
        window_times = times[start - 1:start + gap_size + 1]

        # Exclude windows that already have a break longer than 30 minutes.
        if np.any(np.diff(window_times) > 30 * 60):
            continue

        hidden_times = [as_datetime(t) for t in times[start:start + gap_size]]
        actual = co2[start:start + gap_size]

        for method in ("previous", "linear"):
            begin = perf_counter()
            predicted = reconstruct_gap(
                as_datetime(times[start - 1]),
                co2[start - 1],
                hidden_times,
                as_datetime(times[start + gap_size]),
                co2[start + gap_size],
                method,
            )
            runtimes[method] += perf_counter() - begin
            errors[method].extend(actual - np.array(predicted))

        gap_count += 1

    print(f"\n{gap_size} hidden readings | {gap_count} tested gaps")
    for method in ("previous", "linear"):
        error = np.array(errors[method])
        if len(error):
            mae = np.mean(np.abs(error))
            rmse = np.sqrt(np.mean(error ** 2))
            print(
                f"{method}: MAE {mae:.2f} ppm | "
                f"RMSE {rmse:.2f} ppm | "
                f"method time {runtimes[method]:.4f} s"
            )


6 hidden readings | 246 tested gaps
previous: MAE 13.20 ppm | RMSE 17.45 ppm | method time 0.0016 s
linear: MAE 9.79 ppm | RMSE 12.50 ppm | method time 0.0024 s

18 hidden readings | 132 tested gaps
previous: MAE 18.42 ppm | RMSE 25.36 ppm | method time 0.0011 s
linear: MAE 12.45 ppm | RMSE 16.20 ppm | method time 0.0023 s

36 hidden readings | 78 tested gaps
previous: MAE 27.30 ppm | RMSE 38.00 ppm | method time 0.0009 s
linear: MAE 15.94 ppm | RMSE 21.00 ppm | method time 0.0021 s
